In [1]:
import folium
from folium.plugins import MarkerCluster, HeatMap
import pandas as pd


In [5]:

# Load your dataset
df = pd.read_csv("/Users/renflores/Documents/GFC/agb-ref-data-processing/data/output/brazil_nfi_2018_validation_data.csv")  # adjust this to your file
# Expected columns: 'latitude', 'longitude', 'biomass', 'stddev'

In [6]:
df.columns

Index(['PLOT_ID', 'AGB', 'SIZE_HA', 'GEZ', 'AVG_YEAR', 'POINT_X', 'POINT_Y',
       'ZONE', 'sdTree', 'AGB_T_HA_ORIG', 'sdGrowth', 'sdSE', 'varPlot', 'VER',
       'sdMap', 'BIO', 'REALM', 'OPEN', 'TIER'],
      dtype='object')

In [24]:
df.describe()

,PLOT_ID,biomass,SIZE_HA,AVG_YEAR,longitude,latitude,stddev,AGB_T_HA_ORIG,sdGrowth,sdSE,varPlot,VER,sdMap,OPEN,TIER
count,7684.000000,7684.000000,7684.0,7684.000000,7684.000000,7684.000000,7684.000000,7684.000000,7684.000000,7.684000e+03,7684.000000,7684.0,0.0,7684.0,0.0
mean,524.245445,450.773843,0.5,2017.510021,-56.680531,-5.452686,28.892970,450.773843,4.344977,5.363575e+01,3858.605331,7.0,NaN,0.0,NaN
std,382.641274,1578.031543,0.0,2.722160,7.068350,4.379777,10.693152,1578.031543,3.715137,2.131767e-14,822.501844,0.0,NaN,0.0,NaN
min,1.000000,0.393295,0.5,2014.000000,-73.081109,-14.579069,20.478441,0.393295,0.000000,5.363575e+01,3296.160256,7.0,NaN,0.0,NaN
25%,218.000000,61.229833,0.5,2015.000000,-61.021396,-9.000912,22.479082,61.229833,0.000000,5.363575e+01,3406.661176,7.0,NaN,0.0,NaN
50%,458.000000,126.413168,0.5,2018.000000,-58.681417,-6.120672,23.928567,126.413168,5.000000,5.363575e+01,3469.472804,7.0,NaN,0.0,NaN
75%,724.000000,338.786641,0.5,2018.000000,-50.040332,-2.338861,34.353646,338.786641,7.500000,5.363575e+01,4072.069676,7.0,NaN,0.0,NaN
max,1557.000000,45799.871572,0.5,2024.000000,-43.919434,4.140900,60.754232,45799.871572,15.000000,5.363575e+01,6711.870361,7.0,NaN,0.0,NaN


In [7]:
df.rename(columns={'POINT_Y': 'latitude', 'POINT_X': 'longitude', 'AGB': 'biomass', 'sdTree': 'stddev'}, inplace=True)

In [27]:
import folium
from folium.plugins import MarkerCluster, HeatMap

# Center the map
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

# Create base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# ---- 1. Cluster Map of Plot Density ----
marker_cluster = MarkerCluster(name="Plot Density").add_to(m)
for row in df.itertuples():
    folium.Marker(
        location=[row.latitude, row.longitude],
        popup=f"Biomass: {row.biomass}\nStdDev: {row.stddev}"
    ).add_to(marker_cluster)

# ---- 2. Heatmap of Biomass ----
biomass_heat_data = [
    [row.latitude, row.longitude, row.biomass] for row in df.itertuples()
]
HeatMap(
    biomass_heat_data,
    name="Biomass Heatmap",
    min_opacity=0.5,
    radius=15,
    blur=10
).add_to(m)

# ---- 3. Heatmap of Biomass Standard Deviation ----
stddev_heat_data = [
    [row.latitude, row.longitude, row.stddev] for row in df.itertuples()
]
# HeatMap(
#     stddev_heat_data,
#     name="Biomass StdDev Heatmap",
#     min_opacity=0.5,
#     radius=15,
#     blur=10,
#     gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}
# ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save to HTML
m.save('./index.html')